# OrganicAI — 협력 층위(L0~L4) 판정 파이프라인

시나리오 하나를 넣으면 두 에이전트의 협력 층위를 판정한다.

**처음 쓴다면 `docs/SCENARIO_TEST.md`를 먼저 읽을 것.** (API 키 발급 · Colab 설정 · 에러 대처)

## 실행 순서 요약

| 단계 | 언제 |
|---|---|
| 0. 설치 | 세션 최초 1회 |
| 1~3 | 매 세션 필수 (순서 지킬 것) |
| 4~5 | 시나리오를 새로 썼을 때 |
| 6 | 데이터 수집 |

> ⚠️ **런타임을 재시작했다면 0번을 건너뛰고 1번부터 다시 실행한다.**
> `!git pull` 로 코드를 받은 뒤에도 **반드시 런타임을 재시작**해야 새 코드가 반영된다.

## 0. 설치 (세션 최초 1회)

In [ ]:
!git clone -b yr https://github.com/ewha-oi/OrganicAI.git
%cd OrganicAI
!pip install -r requirements.txt -q

import sys
sys.path.append('src')
print("Python:", sys.version)

## 1. 세션 시작

**런타임 재시작 후에는 여기서부터 실행한다.**

In [ ]:
%cd /content/OrganicAI
import sys
sys.path.append('src')
!git log --oneline -1

In [ ]:
from google.colab import userdata

API_KEYS = {
    "gemini": userdata.get('GEMINI_API_KEY'),   # 현재 미사용 - '없음'이어도 정상
    "groq":   userdata.get('GROQ_API_KEY'),     # 필수
}

for k, v in API_KEYS.items():
    print(f"{k:8s} {'OK' if v else '!! 없음 - Secrets 이름/노트북 액세스 토글 확인'}")

In [ ]:
# 내 Groq 키로 지금 쓸 수 있는 모델의 특성을 조사한다. 모델당 1~2회, 짧은 호출.
# 아래 셀(judge 설정)과 3번(alpha 대체)에 무엇을 넣을지 정하는 근거를 만든다.
#
# 표 읽는 법
#   JSON   response_format 지원 여부. judge 는 O 가 아니면 안 된다 (llm.py 가 400 을 받는다).
#   한글   응답의 한글 비율. 낮으면 한국어 지시를 무시하는 것 -> alpha 부적합.
#   비고   <think> 누출은 추론 과정이 그대로 채점 대상이 된다는 뜻 -> 생성 역할 탈락.
#   ctx    alpha 는 대화 전체 + 마무리 프롬프트를 받으므로 큰 쪽이 좋다.
#   TPM    무료 티어 분당 토큰 한도. max_turns 상한을 이것이 정한다.
#
# 배정은 judge -> alpha -> beta 순으로 고른다 (채점이 무너지면 실험이 성립하지 않는다).
# 셋의 계열이 전부 달라야 한다 — llm.py 의 self-preference 편향 관련 조항 참고
import re, time
from groq import Groq

client   = Groq(api_key=API_KEYS["groq"])
NOT_CHAT = ("whisper", "tts", "embed", "guard", "moderation", "ocr")
FAMILIES = ("gemini","gemma","claude","llama","gpt","qwen","deepseek",
            "mistral","kimi","moonshot","grok","compound")
PROBE = ('한국어로만 답하라. 교내 에너지 절약 캠페인 기획안을 3문장으로 요약해 '
         '아래 JSON 형식으로만 출력하라. {"제목": "...", "요약": "..."}')

rows, bad = [], []
for m in sorted(client.models.list().data, key=lambda x: x.id):
    if any(t in m.id.lower() for t in NOT_CHAT):
        continue
    low = m.id.lower()
    r = {"id": m.id, "ctx": getattr(m, "context_window", 0) or 0,
         "fam": next((f for f in FAMILIES if f in low), low.split("/")[-1].split("-")[0])}
    t0 = time.time()
    for json_mode in (True, False):      # JSON 모드 지원 여부 = judge 자격 요건
        kw = {"response_format": {"type": "json_object"}} if json_mode else {}
        try:
            resp = client.chat.completions.with_raw_response.create(
                model=m.id, messages=[{"role": "user", "content": PROBE}],
                max_tokens=2048, temperature=0, **kw)
            text  = resp.parse().choices[0].message.content or ""
            dense = re.sub(r"\s", "", text)
            r.update(json_mode=json_mode, sec=round(time.time()-t0, 1),
                     ko=round(len(re.findall(r"[가-힣]", text))/max(len(dense),1), 2),
                     think="<think>" in text, empty=not dense,
                     tpm=resp.headers.get("x-ratelimit-limit-tokens", "?"),
                     rpd=resp.headers.get("x-ratelimit-limit-requests", "?"),
                     head=re.sub(r"\s+", " ", text)[:60])
            rows.append(r); break
        except Exception as e:
            err = str(e)[:70]
    else:
        bad.append({"id": m.id, "err": err})
    print(".", end="")

print(f"\n\n{'모델':42s}{'계열':9s}{'ctx':>8s}{'JSON':>6s}{'한글':>6s}"
      f"{'초':>6s}{'TPM':>8s}{'RPD':>7s}  비고")
print("-" * 100)
for r in rows:
    note = ("<think>누출 " if r["think"] else "") + ("빈응답(토큰부족)" if r["empty"] else "")
    print(f"{r['id'][:41]:42s}{r['fam'][:8]:9s}{r['ctx']:>8d}"
          f"{('O' if r['json_mode'] else 'X'):>6s}{r['ko']:>6.2f}{r['sec']:>6.1f}"
          f"{str(r['tpm']):>8s}{str(r['rpd']):>7s}  {note}")

print("\n[이 키로 호출 불가 — 목록에는 있으나 티어/퇴역]")
for r in bad:
    print(f"  {r['id'][:41]:42s}{r['err']}")

print("\n[샘플 출력 — 한국어와 형식을 눈으로 볼 것]")
for r in rows:
    print(f"  {r['id'][:41]:42s}{r['head']}")


In [ ]:
# 모델 배정 — 아래 1-2 절의 실측으로 확정한 값 (2026-08-23).
#
#   judge = qwen/qwen3.6-27b      태깅 4/5
#   alpha = openai/gpt-oss-120b   태깅 4/5 (여기서는 생성 역할로 쓴다)
#   beta  = openai/gpt-oss-20b    _call_llama 경로 실측 OK (346자 / 1.8초)
#
# judge 만 qwen 계열인 이유: alpha·beta 가 둘 다 gpt 계열이라 judge 까지 gpt 면
# 자기 계열이 쓴 산출물을 자기가 채점하게 된다 (self-preference 편향).
# 이 계정에 쓸 수 있는 계열이 gpt·qwen 둘뿐이라 어딘가는 겹칠 수밖에 없는데,
# 겹치는 자리를 '측정(채점)'이 아니라 '설정(생성)' 쪽에 두는 선택이다.
#
# 반드시 coop_pipeline 을 임포트하는 어떤 셀보다 먼저 실행할 것 —
# llm.py 가 임포트 시점에 이 값을 한 번만 읽는다.
# 값을 바꾸려면 여기서 고치고 런타임을 재시작해야 반영된다.

import os
os.environ["COOP_JUDGE_PROVIDER"] = "groq"                 # anthropic / groq / gemini
os.environ["COOP_JUDGE_MODEL"]    = "qwen/qwen3.6-27b"
os.environ["COOP_ALPHA_MODEL"]    = "openai/gpt-oss-120b"
os.environ["COOP_BETA_MODEL"]     = "openai/gpt-oss-20b"   # 기본값 llama-3.3-70b 는 이 계정에 없다

for k in ("COOP_JUDGE_PROVIDER", "COOP_JUDGE_MODEL",
          "COOP_ALPHA_MODEL", "COOP_BETA_MODEL"):
    print(f"{k:22s} {os.environ[k]}")


## 1-2. 모델 배정 실측 (배정을 확정할 때 1회만)

alpha / beta / judge 를 무엇으로 고정할지 **실측으로** 정한다.
한 번 확정해 위 셀에 적어 넣고 나면 이 절은 다시 돌릴 필요가 없다.

> ⚠️ 아래 두 셀은 `coop_pipeline` 을 임포트한다.
> 결과를 보고 **위 셀(judge 설정)의 값을 고쳤다면 반드시 런타임을 재시작**해야 반영된다.

### 지금까지 측정된 것

| 모델 | 계열 | judge 태깅 | 비고 |
|---|---|---|---|
| `openai/gpt-oss-120b` | gpt | **4/5** | `comp` 를 놓침 — `comp_ratio_min` 이 Q4a 에 걸려 있으므로 동결 전 재확인 |
| `openai/gpt-oss-20b` | gpt | 3/5 | 과잉 태깅 (phatic 을 lead 로) |
| `qwen/qwen3.6-27b` | qwen | 미측정 | `llm.py` 가 400 을 받아 막힘 → 아래 `[1]` 로 우회 측정 |
| `groq/compound` | compound | - | RPD 250, 내부 도구 호출 → 대화에 외부 정보가 섞일 수 있음 |
| `allam-2-7b` | allam | - | ctx 4096 < `_MIN_OUTPUT_TOKENS` 4096 → 구조적으로 사용 불가 |

**`llama-3.3-70b-versatile` 은 이 계정에 없다.** `MODELS["beta"]` 의 기본값이므로
그대로 두면 beta 턴마다 404 가 난다. 반드시 교체해야 한다.

### 배정 결정 규칙

쓸 수 있는 계열이 `gpt` 와 `qwen` 둘뿐이라 어딘가는 반드시 겹친다. `[1]` 의 결과가 정한다.

| `[1]` 결과 | judge | alpha | beta | 겹치는 곳 |
|---|---|---|---|---|
| qwen ≥ 4/5 | `qwen3.6-27b` | `gpt-oss-120b` | `gpt-oss-20b` | alpha ≡ beta — **채점 편향 없음** |
| qwen < 4/5 | `gpt-oss-120b` | `qwen3.6-27b` | `gpt-oss-20b` | judge ≡ beta — beta 단독 점수만 부풀어 Q3 가 **어려워지는**(보수적) 방향 |

In [ ]:
# [1] judge 후보의 태깅 정확도. 4/5 이상이 통과 기준 (4번 절과 같은 사례).
#
# llm.py 를 거치지 않고 같은 조건(system=CODING_MANUAL, JSON 모드, temperature=0)을
# 직접 재현한다. 우회하는 이유:
#   llm.py:167 의 _GROQ_REASONING_MODELS 에 "qwen3" 이 들어 있어 qwen 에
#   reasoning_effort="low" 를 보내는데, qwen3.6 은 'none'/'default' 만 받는다 -> 400.
#   그래서 llm.py 를 그냥 쓰면 qwen 은 능력과 무관하게 0/5 가 나온다.
# 여기서는 모델마다 맞는 값을 보내 '태깅 능력'만 분리해서 잰다.

from coop_pipeline.tagging import CODING_MANUAL
from coop_pipeline.llm import parse_json_strict
from groq import Groq

CANDIDATES = {
    "qwen/qwen3.6-27b":      "none",   # 'low' 를 안 받는다
    # "openai/gpt-oss-120b": "low",    # 이미 4/5. 다시 재려면 주석 해제
    # "openai/gpt-oss-20b":  "low",    # 이미 3/5
}

CASES = [
    ("수요일 B실로 하자.",     "좋아, 그렇게 하자.",                                 "phatic"),
    ("수요일에 하는 게 어때?",  "맞네, 나는 A실을 생각했는데 수요일이면 B실이 맞겠다.",  "agree"),
    ("수요일 B실로 하자.",     "좋아. 그런데 예산 확인도 필요해 보여.",                "comp"),
    ("예산은 200이야.",       "응, 200이지.",                                     "phatic"),
    ("회의 준비 시작하자.",    "지금 정할 건 요일이야. 시간은 나중에.",                "lead"),
]

client = Groq(api_key=API_KEYS["groq"])
for model_id, effort in CANDIDATES.items():
    print(f"=== {model_id}  (reasoning_effort={effort})")
    hit = 0
    for prev, cur, want in CASES:
        try:
            r = client.chat.completions.create(
                model=model_id, temperature=0, max_tokens=4096,
                reasoning_effort=effort,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": CODING_MANUAL},
                          {"role": "user", "content":
                           f"대화 맥락:\nalpha: {prev}\n\n분류할 발화 (beta): {cur}"}])
            text = r.choices[0].message.content or ""
            if not text:
                print(f"  X 기대={want:6s} 빈 응답 — 추론 토큰 소진")
                continue
            codes = parse_json_strict(text).get("codes", [])
        except Exception as e:
            print(f"  X 기대={want:6s} {str(e)[:80]}")
            continue
        ok = (codes == ["phatic"]) if want == "phatic" else (want in codes)
        hit += ok
        print(f"  {'O' if ok else 'X'} 기대={want:6s} 실제={codes}")
    print(f"  -> {hit}/5\n")


In [ ]:
# [2] beta 경로 점검.
#
# agents._call_llama (agents.py:83) 는 max_tokens 도 reasoning_effort 도 보내지 않는다.
# gpt-oss 는 추론 모델이라 3번 절의 alpha 와 똑같이 빈 응답이 날 수 있다.
# 추측하지 말고 실제 호출 경로를 그대로 태워서 확인한다.
#
#   OK 가 뜨면  -> beta 는 코드 수정 없이 쓸 수 있다.
#   실패하면    -> _call_llama 에 추론 옵션을 넣는 src 수정이 필요하다.

import time
from groq import Groq
from coop_pipeline import agents, llm
from coop_pipeline.runner import load_scenario

BETA_CANDIDATE = "openai/gpt-oss-20b"

llm.MODELS["beta"] = BETA_CANDIDATE
v = load_scenario("scenarios/A4/A4_simple_energy_campaign.json")["task_variants"]
prompt = agents.SYSTEM_PROMPT_TEMPLATES["명시"].format(
    name="beta", partner="alpha", task=v.get("beta") or v["shared"]
) + "\n\n지금까지의 대화:\n(없음)\n\n너의 다음 발언:"

t0 = time.time()
try:
    out = agents._call_llama(Groq(api_key=API_KEYS["groq"]), prompt)
    print(f"OK — {len(out)}자 / {time.time() - t0:.1f}초\n")
    print(out[:400])
except Exception as e:
    # with_retry 가 3회 재시도하므로 실패 시 10초 안팎 걸린다.
    print(f"실패 ({time.time() - t0:.1f}초):", str(e)[:250])


## 2. 환경 점검 (API 호출 없음, 무료)

여기서 걸리는 문제는 실행해도 똑같이 걸린다. 먼저 통과시킬 것.

In [ ]:
!python -m pytest tests/ -q
!python tools/dryrun_frame.py

In [ ]:
# 시나리오 형식 + 현재 모델 구성 확인
from coop_pipeline.runner import check_scenario_dir
from coop_pipeline.llm import MODELS, judge_provider, judge_model, judge_key_name

check_scenario_dir("scenarios")

print(f"\nalpha : {MODELS['alpha']}")
print(f"beta  : {MODELS['beta']}")
print(f"judge : {judge_provider()}:{judge_model()}   (필요한 키: {judge_key_name()})")

## 3. [임시] alpha 모델 대체

Google이 Gemini API 접근을 차단해서(전 모델 403/404) alpha를 Groq 모델로 돌린다.

**Gemini가 복구되면 이 셀만 실행하지 않으면 원래 설계로 돌아간다. 코드 수정 불필요.**
그때는 `COOP_ALPHA_MODEL` 환경변수로 모델 ID만 지정하면 된다.

> 이 대체 때문에 alpha·beta·judge가 모두 Groq 무료 티어를 쓴다.
> 분당 토큰 한도(TPM)에 걸리기 쉬우므로 `max_turns` 를 6 이하로 두는 것이 안전하다.

In [ ]:
import re, types
from groq import Groq
from coop_pipeline import agents, llm
from coop_pipeline.runner import load_scenario

GROQ_KEY = API_KEYS["groq"]

# 하드코딩하지 말 것. 위 1번 절의 COOP_ALPHA_MODEL 이 유일한 출처다.
# (예전에 여기 모델 ID를 직접 박아두어, 환경변수를 바꿔도 반영되지 않는 일이 있었다.)
ALPHA_MODEL = llm.MODELS["alpha"]
_THINK = re.compile(r"<think>.*?</think>\s*", re.S)

# 이 모델이 어떤 추론 옵션을 받는지 먼저 확인한다 (모델마다 다르고, 틀리면 400).
#   gpt-oss   : reasoning_effort = low / medium / high
#   qwen3.6   : reasoning_effort = none / default   ← low 를 보내면 400
# 'none' 은 추론을 아예 끄므로, 추론 토큰이 max_tokens 를 다 먹고 본문이 비는 것도 막는다.
EXTRA = {}
for cand in ({"reasoning_effort": "low",  "reasoning_format": "hidden"},
             {"reasoning_effort": "none", "reasoning_format": "hidden"},
             {"reasoning_effort": "low"},
             {"reasoning_effort": "none"},
             {"reasoning_format": "hidden"},
             {}):
    try:
        Groq(api_key=GROQ_KEY).chat.completions.create(
            model=ALPHA_MODEL, messages=[{"role": "user", "content": "ping"}],
            max_tokens=64, **cand)
        EXTRA = cand
        break
    except Exception as e:
        print("불가:", cand, "|", str(e)[:90])
print(f"alpha = {ALPHA_MODEL}")
print("사용할 옵션:", EXTRA, "\n")


class _Resp:
    def __init__(self, text): self.text = text


class _GroqModel:
    def __init__(self, model_id): self.model_id = model_id

    def generate_content(self, prompt):
        r = Groq(api_key=GROQ_KEY).chat.completions.create(
            model=self.model_id,
            messages=[{"role": "user", "content": prompt}],
            temperature=agents.TEMPERATURE,
            max_tokens=4096,          # 없으면 추론 토큰에 다 쓰고 빈 응답이 온다
            **EXTRA,
        )
        text = _THINK.sub("", r.choices[0].message.content or "").strip()
        if not text:
            # 원인은 둘 중 하나다: (a) max_tokens 부족, (b) 추론이 출력 한도를 다 먹음.
            # (b)면 위 EXTRA 에서 reasoning_effort='none'(또는 'low')이 잡혔는지 확인할 것.
            raise RuntimeError(
                f"alpha({self.model_id}) 빈 응답 — max_tokens 부족이거나 "
                f"추론이 출력 한도를 소진함. 현재 EXTRA={EXTRA}")
        return _Resp(text)


_shim = types.SimpleNamespace(configure=lambda **kw: None, GenerativeModel=_GroqModel)
agents._gemini_model = lambda: _shim
agents.RATE_LIMIT_SLEEP = 3.0   # TPM 8000. 429 가 뜨면 6.0~10.0 으로 올릴 것

# 스모크 테스트는 반드시 '실제 길이의' 프롬프트로 한다.
# 짧은 ping은 통과해도 실제 프롬프트에서 토큰 한도에 걸리는 일이 있다.
sc = load_scenario("scenarios/A4/A4_simple_energy_campaign.json")
v = sc["task_variants"]
task = v.get("alpha") or v["shared"]      # 비대칭/대칭 시나리오 양쪽 모두 대응
long_prompt = agents.SYSTEM_PROMPT_TEMPLATES["명시"].format(
    name="alpha", partner="beta", task=task
) + "\n\n지금까지의 대화:\n(없음)\n\n너의 다음 발언:"

out = _GroqModel(ALPHA_MODEL).generate_content(long_prompt).text
print(f"길이 {len(out)}자")
print(out[:300])
print("\nMODELS:", llm.MODELS)


## 4. 발화 태깅 점검

채점자가 발화 코드(phatic/meta/lead/arch/agree/comp)를 제대로 붙이는지 5개 사례로 본다.
**4/5 이상이면 통과.** 3/5 이하면 채점자 모델을 의심할 것.

In [ ]:
from coop_pipeline.llm import make_judge
from coop_pipeline.tagging import tag_turn

CASES = [
    ("수요일 B실로 하자.",     "좋아, 그렇게 하자.",                                 "phatic"),
    ("수요일에 하는 게 어때?",  "맞네, 나는 A실을 생각했는데 수요일이면 B실이 맞겠다.",  "agree"),
    ("수요일 B실로 하자.",     "좋아. 그런데 예산 확인도 필요해 보여.",                "comp"),
    ("예산은 200이야.",       "응, 200이지.",                                     "phatic"),
    ("회의 준비 시작하자.",    "지금 정할 건 요일이야. 시간은 나중에.",                "lead"),
]

j = make_judge(API_KEYS["groq"])
hit = 0
for prev, cur, want in CASES:
    got = tag_turn(j, [{"turn": 1, "speaker": "alpha", "text": prev}],
                   {"turn": 2, "speaker": "beta", "text": cur})
    ok = (got["codes"] == ["phatic"]) if want == "phatic" else (want in got["codes"])
    hit += ok
    print(f"{'O' if ok else 'X'} 기대={want:6s} 실제={got['codes']} ref={got['ref']}")
    print(f"   근거: {got['evidence']}")
print(f"\n{hit}/5")

## 5. 시나리오 실기동

**새 시나리오를 썼다면 여기서 한 번 돌려본다.**

통과 기준은 하나뿐이다 — **에러 없이 `L0`~`L4` 중 하나가 나오는 것.**
어느 층위가 나오든, `[FAIL] Q3` 이 뜨든 상관없다. 그건 실험 결과이지 형식 오류가 아니다.

In [ ]:
# 시나리오 경로만 본인 파일로 바꿔서 쓴다.
from coop_pipeline.runner import run_scenario

result = run_scenario(
    "scenarios/A4/A4_simple_energy_campaign.json",
    condition="명시",     # 또는 "묵시"
    api_keys=API_KEYS,
    n_solo=2,            # 시험용. 정식 실행은 5
    max_turns=6,         # 시험용. 정식 실행은 10 (단, 임시 alpha는 TPM 한도 주의)
    out_dir="runs_pilot",
)

In [ ]:
# 무엇이 실제로 일어났는지 들여다보기 (에러 원인 추적용)
log = result["log"]

print("=== 1) 대화가 실제로 오갔는가 ===")
for t in log["turns"]:
    print(f"[{t['turn']}] {t['speaker']}: {t['text'][:150]}")

print("\n=== 2) 태깅이 붙었는가 ===")
for t in log["turns"]:
    print(t["turn"], t["speaker"], t["codes"], "ref:", t["ref"])

print("\n=== 3) 무엇을 채점했는가 ===")
print(log["group_output_text"][:500])
print("\n단독:", log.get("solo_grades") or log.get("solo_scores"),
      "/ 그룹:", log.get("group_grade") or log.get("group_score"))

In [ ]:
# A1 경로 점검 — 비대칭 지문(alpha/beta) + 체크리스트 채점 + 퍼센타일 판정.
# A2(shared)와는 다른 코드 경로를 타므로 별도로 한 번 확인한다.
from coop_pipeline.runner import run_scenario

res_A1 = run_scenario(
    "scenarios/A1/A1_simple_meeting.json",
    condition="명시",
    api_keys=API_KEYS,
    n_solo=2,
    max_turns=6,
    out_dir="runs_pilot",
)

print("\n판정 :", res_A1["level"])
print("병목 :", res_A1["stopped_at"] or "없음 (L4까지 통과)")

In [ ]:
# 저장된 로그를 다시 판정한다. API 호출 0, 무료.
!ls -R runs_pilot

from coop_pipeline.runner import classify_saved_dir
_ = classify_saved_dir("runs_pilot")

## 6. 정식 실행

명시/묵시 두 조건을 돌린다. 단독 산출물은 한 번만 만들어 두 조건이 공유하므로
조건 간 비교의 기준선이 통일된다.

> ⚠️ 임시 alpha(Groq 무료 티어)로 `max_turns=10` 을 쓰면 분당 토큰 한도에 걸릴 수 있다.
> `413 Request too large` 가 뜨면 `max_turns` 를 낮출 것.
> 대화 원본은 `runs/raw/` 에 태깅 전에 저장되므로 뒤 단계에서 실패해도 유실되지 않는다.

In [ ]:
from coop_pipeline.runner import run_scenario_both_conditions

results = run_scenario_both_conditions(
    "scenarios/A4/A4_simple_energy_campaign.json",
    api_keys=API_KEYS, n_solo=5, max_turns=10, out_dir="runs",
)
print(results["명시"]["level"], results["묵시"]["level"])

In [ ]:
# Drive 백업 — 실행이 끝나면 바로 할 것. Colab 세션이 끊기면 runs/ 는 사라진다.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/OrganicAI_runs
!cp -r runs/* /content/drive/MyDrive/OrganicAI_runs/
!ls /content/drive/MyDrive/OrganicAI_runs/

In [ ]:
# 임계값을 바꿔가며 판정이 어떻게 달라지는지 본다. API 호출 0, 무료.
#
# 주의: 결과를 보고 나서 결과에 맞춰 기준을 고치면 안 된다.
#       여기서는 '어느 기준이 결과를 좌우하는지' 관찰만 하고,
#       확정은 데이터를 충분히 모은 뒤 configs/thresholds_v2.json 으로 한다.
from coop_pipeline.runner import classify_saved_dir
from coop_pipeline import load_thresholds

for gap in (2, 1.5, 1, 0.5):
    print(f"\n### grade_gap_min = {gap}")
    classify_saved_dir("runs", dict(load_thresholds("v1"), grade_gap_min=gap))

## 부록

In [ ]:
# 최신 코드 받아오기. 받은 뒤에는 반드시 런타임을 재시작하고 1번부터 다시 실행할 것.
!git pull
!git log --oneline -3

In [ ]:
# Groq에서 지금 쓸 수 있는 모델 목록.
# 모델이 퇴역해 실행이 통째로 실패할 때 여기서 대체 ID를 고른다.
# 규칙: judge 는 alpha·beta 어느 쪽과도 다른 계열이어야 한다 (self-preference 편향).
from groq import Groq

for m in sorted(x.id for x in Groq(api_key=API_KEYS["groq"]).models.list().data):
    print("  ", m)